# MedAESQA Data Prep — FaithfulMed (Google 2D)

Owner: Isha and Isabella | September milestone, Task #3

**What this notebook produces.** Four clean CSV files plus a freeze manifest, so nobody
else on the team ever has to open the raw JSON or re-derive our cleaning rules.

| File | One row per | Used by |
|---|---|---|
| `medaesqa_questions.csv` | question (40) | everyone |
| `medaesqa_nuggets.csv` | expert fact (498) | Extractor |
| `medaesqa_sentences.csv` | answer sentence (5,162) | Verifier, Readability |
| `medaesqa_citations.csv` | sentence-PMID pair (~8,124) | Verifier |
| `medaesqa_freeze.txt` | — | reproducibility |

**What this notebook does NOT do.** No imputation, no normalization, no outlier removal.
There are no numeric features here to normalize and no missing values to fill. Everything
in MedAESQA is text and human labels. The real prep work is flattening nested JSON,
standardizing inconsistent category strings, and fixing one genuine ambiguity in how
citations are counted.

**Prerequisite.** `medaesqa_v1.json` in the same folder. Download from https://osf.io/ydbzq/

## 1. Load and fingerprint

The hash is the whole point of freezing. If it changes, every number the team has
reported is suspect and we need to know immediately.

In [12]:
!pip install -q pandas

In [13]:
import json
import re
import hashlib
from pathlib import Path

import pandas as pd

RAW_PATH = Path("medaesqa_v1.json")
OUT_DIR = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

SOURCE_SHA256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()

with open(RAW_PATH) as f:
    raw = json.load(f)

items = raw if isinstance(raw, list) else list(raw.values())

print("source sha256:", SOURCE_SHA256)
print("questions loaded:", len(items))

source sha256: a062742c1678b4379121ad030a06bda574afd376f331df41575938dd9849c383
questions loaded: 40


## 2. Field names

These are the real key names, confirmed against the file during EDA. They are written out
explicitly rather than guessed at, so that a rename upstream fails loudly instead of
silently producing empty columns.

In [14]:
# question_frame
K_FOCUS = "Focus"
K_TYPE = "Type"
K_TASK = "Task"
K_SUBJECT = "Subject matter (focus)"
K_BODY = "Body system"
K_SPECIALTY = "Specialty"

# machine answers
K_ACCURATE = "is_answer_accurate"
K_SENTENCES = "answer_sentences"
K_SENT_ID = "answer_sentence_id"
K_SENT_TEXT = "answer_sentence"
K_SENT_REL = "answer_sentence_relevance"
K_CITES = "citation_assessment"
K_PMID = "cited_pmid"
K_EXCERPT = "evidence_support"
K_RELATION = "evidence_relation"


def check_keys(sample, keys, label):
    """Fail loudly if the file no longer has the fields we expect."""
    missing = [k for k in keys if k not in sample]
    if missing:
        raise KeyError(f"{label}: expected keys not found: {missing}")
    print(f"{label}: all {len(keys)} expected keys present")


first = items[0]
check_keys(first["question_frame"], [K_FOCUS, K_TYPE, K_TASK, K_SUBJECT, K_BODY, K_SPECIALTY],
           "question_frame")

first_method = next(iter(first["machine_generated_answers"].values()))
check_keys(first_method, [K_ACCURATE, K_SENTENCES], "machine answer")
check_keys(first_method[K_SENTENCES][0], [K_SENT_ID, K_SENT_TEXT, K_SENT_REL, K_CITES],
           "answer sentence")

question_frame: all 6 expected keys present
machine answer: all 2 expected keys present
answer sentence: all 4 expected keys present


## 3. Questions

One cleaning decision here. `Specialty` and `Body system` hold multiple values in a single
string with inconsistent spacing, e.g. `pulmonology/ ENT / cardiology`. Counting those as
one category makes cardiology look rarer than it is.

We split on `/`, strip whitespace, and lowercase. The original strings are kept in
`*_raw` columns so nobody has to take our cleaning on faith.

In [15]:
def split_multi(value):
    """'pulmonology/ ENT / cardiology' -> ['pulmonology', 'ent', 'cardiology']"""
    if not value:
        return []
    return [p.strip().lower() for p in str(value).split("/") if p.strip()]


q_rows = []
for it in items:
    frame = it.get("question_frame", {}) or {}
    q_rows.append({
        "question_id": it.get("question_id"),
        "question": it.get("question", ""),
        "question_focus": frame.get(K_FOCUS),
        "question_type": frame.get(K_TYPE),
        "task": frame.get(K_TASK),
        "subject_matter": frame.get(K_SUBJECT),
        "body_system_raw": frame.get(K_BODY),
        "specialty_raw": frame.get(K_SPECIALTY),
        "body_systems": split_multi(frame.get(K_BODY)),
        "specialties": split_multi(frame.get(K_SPECIALTY)),
        "expert_answer": (it.get("expert_curated_answer") or "").strip(),
    })

questions = pd.DataFrame(q_rows)
questions["n_specialties"] = questions["specialties"].str.len()
questions["n_body_systems"] = questions["body_systems"].str.len()

# Lists don't survive a CSV round-trip cleanly, so store them pipe-joined.
questions["specialties"] = questions["specialties"].str.join("|")
questions["body_systems"] = questions["body_systems"].str.join("|")

print(f"{len(questions)} questions")
print(f"{(questions['n_specialties'] > 1).sum()} questions list more than one specialty")
questions.head(3)

40 questions
9 questions list more than one specialty


,question_id,question,question_focus,question_type,task,subject_matter,body_system_raw,specialty_raw,body_systems,specialties,expert_answer,n_specialties,n_body_systems
0,116,Are there ways to prevent sleep apnea or treat...,sleep apnea,Treatment,Decision support,clinical / problem,respiratory,pulmonology/ ENT / cardiology,respiratory,pulmonology|ent|cardiology,There are ways to prevent and treat sleep apne...,3,1
1,117,What will mutation in runx2 affect in the future?,runx2 mutation,Manifestation,Effect,genetics,full body,clinical genetics,full body,clinical genetics,The effect of the runx2 mutation depends on th...,1,1
2,118,how long does a minor cornea injury take to he...,corneal abrasion,Prognosis,Factoid,clinical / problem,visual,ophthalmology,visual,ophthalmology,Most corneal injuries heal in 24 to 72 hours [...,1,1


In [16]:
questions["task"] = questions["task"].str.strip().str.lower()
questions["question_type"] = questions["question_type"].str.strip().str.lower()
questions["subject_matter"] = questions["subject_matter"].str.strip().str.lower()

In [17]:
# Honest specialty counts, after splitting.
exploded = (questions["specialties"].str.split("|").explode().str.strip())
print(exploded.value_counts().head(15))

specialties
cardiology           9
clinical genetics    5
gp                   5
neurology            4
pulmonology          4
endocrinology        3
gyn                  2
ophthalmology        2
ob.                  2
hepatology           2
psychiatry           2
ent                  1
gastroenterology     1
urology              1
rheumatology         1
Name: count, dtype: int64


## 4. Nuggets

Expert facts in `Predicate(subject, object)` form. These are gold for the Extractor.

We parse out the predicate and arguments where the pattern fits, and flag the ones that
don't. That flag matters: it tells the Extractor owner how much schema complexity is real
rather than assumed.

In [18]:
NUGGET_RE = re.compile(r"^\s*([A-Za-z_ ]+?)\s*\((.*)\)\s*$")

nug_rows = []
for it in items:
    qid = it.get("question_id")
    for idx, n in enumerate(it.get("nuggets") or []):
        if isinstance(n, str):
            body, cuis = n, []
        else:
            body = n.get("nugget") or n.get("text") or ""
            cuis = n.get("cuis") or n.get("CUI") or []
            if isinstance(cuis, str):
                cuis = [cuis]

        m = NUGGET_RE.match(body)
        nug_rows.append({
            "question_id": qid,
            "nugget_idx": idx,
            "nugget": body,
            "predicate": m.group(1).strip() if m else None,
            "arguments": m.group(2).strip() if m else None,
            "n_arguments": len(m.group(2).split(",")) if m else None,
            "fits_predicate_pattern": bool(m),
            "cuis": "|".join(str(c) for c in cuis),
            "n_cuis": len(cuis),
        })

nuggets = pd.DataFrame(nug_rows)
print(f"{len(nuggets)} nuggets across {nuggets['question_id'].nunique()} questions")
print(f"{(~nuggets['fits_predicate_pattern']).mean() * 100:.1f}% don't fit Predicate(a, b)")
print(f"{(nuggets['n_cuis'] > 0).mean() * 100:.1f}% carry at least one CUI")
nuggets.head(3)

498 nuggets across 40 questions
34.5% don't fit Predicate(a, b)
0.0% carry at least one CUI


,question_id,nugget_idx,nugget,predicate,arguments,n_arguments,fits_predicate_pattern,cuis,n_cuis
0,116,0,"Exists (treat naturally[C0027495], sleep apnea...",Exists,"treat naturally[C0027495], sleep apnea [C0037315]",2.0,True,,0
1,116,1,"[may] Prevent (lifestyle changes [C0870811], s...",None,None,NaN,False,,0
2,116,2,"[may] Treat (lifestyle changes [C0870811], sle...",None,None,NaN,False,,0


## 5. Sentences

One row per answer sentence. This is the table the Verifier calibrates against.

Two decisions recorded in columns rather than applied destructively:

- **Unlabeled sentences stay in.** 244 sentences carry no relevance grade, and they all
  come from answers already rejected as inaccurate. We flag them with `has_relevance_label`
  instead of dropping them, so whoever computes kappa can exclude them and see why.
- **Nothing is imputed.** A missing label means a human did not grade it. Filling that in
  would be inventing data.

In [19]:
def norm(x):
    return str(x).strip().lower() if x is not None else None


sent_rows = []
for it in items:
    qid = it.get("question_id")
    for method_id, ans in (it.get("machine_generated_answers") or {}).items():
        accurate = norm(ans.get(K_ACCURATE))
        for idx, s in enumerate(ans.get(K_SENTENCES) or []):
            sent_rows.append({
                "question_id": qid,
                "method_id": method_id,
                "sentence_idx": idx,
                "sentence_id": s.get(K_SENT_ID),
                "sentence": (s.get(K_SENT_TEXT) or "").strip(),
                "relevance": norm(s.get(K_SENT_REL)),
                "answer_is_accurate": accurate,
                "n_citations": len(s.get(K_CITES) or []),
            })

sentences = pd.DataFrame(sent_rows)
sentences["has_relevance_label"] = sentences["relevance"].notna()

print(f"{len(sentences)} sentences")
print(f"{sentences['has_relevance_label'].sum()} labeled -> the Verifier calibration pool")
print()
print(sentences["relevance"].value_counts(dropna=False))

5162 sentences
4918 labeled -> the Verifier calibration pool

relevance
required         3958
unnecessary       698
None              244
borderline        243
inappropriate      19
Name: count, dtype: int64


In [20]:
# Confirm the unlabeled sentences really are all from rejected answers.
crosstab = pd.crosstab(sentences["answer_is_accurate"].fillna("(none)"),
                       sentences["has_relevance_label"],
                       rownames=["answer_is_accurate"], colnames=["has_label"])
print(crosstab)

has_label           False  True 
answer_is_accurate              
no                    244      0
yes                     0   4918


## 6. Citations

This is the one place the raw file is genuinely ambiguous, so the rule below is a
decision the team should be able to see and argue with.

A single sentence-PMID pair can appear several times, once per supporting excerpt, and in
390 cases those excerpts carry **different verdicts**. Keeping all rows inflates the
count to 9,611 against the paper's 8,111. Deduplicating arbitrarily picks a verdict at
random.

**Our rule:** collapse to one row per sentence-PMID pair, and when verdicts conflict, the
most severe one wins. Contradiction beats support, because the paper's own definition is
that a document supports a claim only when nothing in it contradicts the claim.

This yields 8,124 citations at 66.8% supporting, against the paper's 8,111 at 67.7%.
The residual gap is unexplained and is on our question list for the advisor, most likely
related to the `invalid citation` label, which appears in the file but not in the paper.

In [21]:
# Most severe first. Anything unrecognized sorts last and is left untouched.
RELATION_PRIORITY = ["contradicts", "supports", "neutral", "not_relevant", "invalid citation"]
RELATION_RANK = {lab: i for i, lab in enumerate(RELATION_PRIORITY)}

# The file uses present participles where the paper uses base forms.
RELATION_MAP = {
    "supporting": "supports",
    "supports": "supports",
    "contradicting": "contradicts",
    "contradicts": "contradicts",
    "neutral": "neutral",
    "not relevant": "not_relevant",
    "not_relevant": "not_relevant",
    "invalid citation": "invalid citation",
}


def norm_relation(x):
    lab = norm(x)
    return RELATION_MAP.get(lab, lab)


cit_rows = []
for it in items:
    qid = it.get("question_id")
    for method_id, ans in (it.get("machine_generated_answers") or {}).items():
        for idx, s in enumerate(ans.get(K_SENTENCES) or []):
            for c in (s.get(K_CITES) or []):
                cit_rows.append({
                    "question_id": qid,
                    "method_id": method_id,
                    "sentence_idx": idx,
                    "pmid": c.get(K_PMID),
                    "evidence_relation": norm_relation(c.get(K_RELATION)),
                    "evidence_excerpt": (c.get(K_EXCERPT) or "").strip(),
                })

citations_raw = pd.DataFrame(cit_rows)
print(f"{len(citations_raw)} raw citation_assessment rows")

PAIR_KEYS = ["question_id", "method_id", "sentence_idx", "pmid"]

conflicts = citations_raw.groupby(PAIR_KEYS)["evidence_relation"].nunique()
n_conflicts = int((conflicts > 1).sum())
print(f"{n_conflicts} pairs carry conflicting verdicts across excerpts")

citations = (citations_raw
             .assign(_rank=citations_raw["evidence_relation"].map(RELATION_RANK).fillna(99))
             .sort_values("_rank")
             .drop_duplicates(PAIR_KEYS)
             .drop(columns="_rank")
             .sort_values(PAIR_KEYS)
             .reset_index(drop=True))

# Keep the excerpt count, since it's information the collapse would otherwise throw away.
excerpt_counts = citations_raw.groupby(PAIR_KEYS).size().rename("n_excerpts")
citations = citations.merge(excerpt_counts, on=PAIR_KEYS, how="left")

print(f"{len(citations)} unique sentence-PMID pairs")
print()
print(citations["evidence_relation"].value_counts(normalize=True).round(3))

9611 raw citation_assessment rows
390 pairs carry conflicting verdicts across excerpts
8124 unique sentence-PMID pairs

evidence_relation
supports            0.668
not_relevant        0.191
neutral             0.069
invalid citation    0.050
contradicts         0.022
Name: proportion, dtype: float64


## 7. Validation

Every number below is checked against the published paper. If any line says FAIL, do not
commit the output, because something upstream has changed.

In [22]:
EXPECTED = {
    "questions": (len(questions), 40),
    "sentences": (len(sentences), 5162),
    "required sentences": (int((sentences["relevance"] == "required").sum()), 3958),
    "evidence excerpts": (int((citations_raw["evidence_excerpt"].str.len() > 0).sum()), 7651),
}

all_ok = True
for name, (got, want) in EXPECTED.items():
    ok = got == want
    all_ok = all_ok and ok
    print(f"{name:22s} got {got:6d}  paper {want:6d}   {'OK' if ok else 'FAIL'}")

print()
print(f"citations             got {len(citations):6d}  paper   8111   "
      f"(known {len(citations) - 8111:+d} gap, see section 6)")
print()
print("ALL EXACT CHECKS PASSED" if all_ok else "SOMETHING CHANGED - investigate before committing")

questions              got     40  paper     40   OK
sentences              got   5162  paper   5162   OK
required sentences     got   3958  paper   3958   OK
evidence excerpts      got   7651  paper   7651   OK

citations             got   8124  paper   8111   (known +13 gap, see section 6)

ALL EXACT CHECKS PASSED


## 8. Write outputs

In [23]:
outputs = {
    "medaesqa_questions.csv": questions,
    "medaesqa_nuggets.csv": nuggets,
    "medaesqa_sentences.csv": sentences,
    "medaesqa_citations.csv": citations,
}

for name, df in outputs.items():
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(f"{name:30s} {len(df):6d} rows  {len(df.columns):2d} cols")

manifest = [
    "MedAESQA processed data - FaithfulMed (Google 2D)",
    "Prepared by: Isha, Isabella",
    "",
    f"source file:   medaesqa_v1.json",
    f"source sha256: {SOURCE_SHA256}",
    "",
    "Row counts:",
]
for name, df in outputs.items():
    manifest.append(f"  {name:32s} {len(df)}")
manifest += [
    "",
    "Cleaning rules applied:",
    "  - Specialty and Body system split on '/', stripped, lowercased.",
    "    Originals kept in *_raw columns.",
    "  - Sentences with no relevance label retained and flagged via",
    f"    has_relevance_label. All {int((~sentences['has_relevance_label']).sum())} "
    "come from answers marked not accurate.",
    "  - Citations collapsed to one row per sentence-PMID pair. On conflicting",
    "    verdicts the most severe wins: contradicts > supports > neutral >",
    f"    not_relevant > invalid citation. {n_conflicts} pairs were affected.",
    "  - No imputation, normalization, or outlier removal. Not applicable to this data.",
    "",
    "Known open question: our citation count is "
    f"{len(citations)} vs the paper's 8111. Likely related to the",
    "'invalid citation' label, which appears in the file but not the paper. Raised with advisor.",
    "",
    "Citation: Gupta, D., Bartels, D. & Demner-Fushman, D. Scientific Data 12, 1035 (2025). CC BY 4.0.",
]

manifest_path = OUT_DIR / "medaesqa_freeze.txt"
manifest_path.write_text("\n".join(manifest))
print()
print(manifest_path.read_text())

medaesqa_questions.csv             40 rows  13 cols
medaesqa_nuggets.csv              498 rows   9 cols
medaesqa_sentences.csv           5162 rows   9 cols
medaesqa_citations.csv           8124 rows   7 cols

MedAESQA processed data - FaithfulMed (Google 2D)
Prepared by: Isha, Isabella

source file:   medaesqa_v1.json
source sha256: a062742c1678b4379121ad030a06bda574afd376f331df41575938dd9849c383

Row counts:
  medaesqa_questions.csv           40
  medaesqa_nuggets.csv             498
  medaesqa_sentences.csv           5162
  medaesqa_citations.csv           8124

Cleaning rules applied:
  - Specialty and Body system split on '/', stripped, lowercased.
    Originals kept in *_raw columns.
  - Sentences with no relevance label retained and flagged via
    has_relevance_label. All 244 come from answers marked not accurate.
  - Citations collapsed to one row per sentence-PMID pair. On conflicting
    verdicts the most severe wins: contradicts > supports > neutral >
    not_relevant > inva

## 9. Data dictionary



**medaesqa_questions.csv** — one row per question (40)
`question_id`, `question`, `question_focus`, `question_type`, `task`, `subject_matter`,
`body_system_raw`, `specialty_raw`, `body_systems` and `specialties` (pipe-joined, cleaned),
`n_specialties`, `n_body_systems`, `expert_answer`

**medaesqa_nuggets.csv** — one row per expert fact (498)
`question_id`, `nugget_idx`, `nugget`, `predicate`, `arguments`, `n_arguments`,
`fits_predicate_pattern`, `cuis`, `n_cuis`

**medaesqa_sentences.csv** — one row per answer sentence (5,162)
`question_id`, `method_id` (M1-M30), `sentence_idx`, `sentence_id`, `sentence`,
`relevance` (required / unnecessary / borderline / inappropriate / null),
`answer_is_accurate`, `n_citations`, `has_relevance_label`

**medaesqa_citations.csv** — one row per sentence-PMID pair (~8,124)
`question_id`, `method_id`, `sentence_idx`, `pmid`, `evidence_relation`
(supports / contradicts / neutral / not_relevant / invalid citation), `evidence_excerpt`,
`n_excerpts`

Join sentences to citations on `question_id` + `method_id` + `sentence_idx`.